# 01. Sessao Spark

Objetivo: criar a sessao Spark sem alterar logs, configuracoes ou ipywidgets.

Entrada: variaveis locais da rotina principal.

Saida: sessao Spark criada.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()

    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="etl-vinculacao-mf-insights",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "SANDBOX": "t2i2016",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        jars=[
            "/dados/shared/bin/ojdbc8.jar",
        ],
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer":
                "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive":"true",
            "spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


# 02. Utilitarios remotos

Objetivo: carregar os recursos ja existentes no ambiente remoto.

Entrada: notebook de utilitarios remotos.

Saida: funcoes e clientes base disponiveis para as proximas etapas.


In [ ]:
try:
    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


# 03. Contexto e clientes

Objetivo: resolver ambiente, destino e clientes ja previstos na rotina principal.

Entrada: variaveis do ambiente Spark.

Saida: database, path_save_hdfs, cliente_db2.


In [ ]:
%%spark

ambiente = ler_variavel_ambiente_spark("AMBIENTE").upper()
dominio = ler_variavel_ambiente_spark("DOMINIO").lower()
hoje = ler_variavel_ambiente_spark("HOJE")

if ambiente == "MODELAGEM":
    sandbox = ler_variavel_ambiente_spark("SANDBOX").lower()
    database = f"sbx_{sandbox}"
    path_save_hdfs = f"/dados/transientes/{dominio}/{sandbox}"
else:
    database = f"hive_{dominio}"
    path_save_hdfs = f"/dados/corporativos/{dominio}"

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(env=env_spark)

# 04. Parametros operacionais

Objetivo: definir constantes tecnicas usadas no pipeline.

Entrada: database e limites JDBC conhecidos.

Saida: nome_tabela e parametros JDBC.


In [ ]:
%%spark

from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

nome_tabela_saida = "pbco_plt_fnc_ctgr"
nome_tabela = f"{database}.{nome_tabela_saida}"

fetchsize_db2 = 10_000
num_partitions_db2 = 16
num_partitions_transacoes = 20
partitions_hive = 10

diagnostico_opt_in = False

# Faixa padrao para colunas de identificacao de cliente
# Usada em CD_CLI, CD_CLI_OCP, COD e F_CLIENTE_COD.
cd_cli_min = 1
cd_cli_max = 999_999_999


# 05. Datas de referencia

Objetivo: calcular a janela mensal de processamento.

Entrada: hoje vindo da sessao Spark.

Saida: data_ini, data_fim e ano_etl.


In [ ]:
%%spark

data_atual = hoje
data_ini = data_atual[:8] + "01"

df_datas = (
    spark.createDataFrame([(1,)], ["id"])
    .withColumn("data_atual", F.lit(data_atual).cast("date"))
    .withColumn("data_ini", F.lit(data_ini).cast("date"))
    .withColumn("data_fim", F.last_day("data_atual"))
    .select("data_atual", "data_ini", "data_fim")
)

linha_datas = df_datas.collect()[0]

data_ini = datetime.strftime(linha_datas[1], "%Y-%m-%d")
data_fim = datetime.strftime(linha_datas[2], "%Y-%m-%d")
ano_etl = datetime.strftime(linha_datas[1], "%Y")

print(f"Data de Inicio da query: {data_ini}")
print(f"Data final da query: {data_fim}")
print(f"Ano do ETL : {ano_etl}")


# 06. Query congelada de transacoes

Objetivo: declarar o SQL legado sem alterar sua logica.

Entrada: data_ini e data_fim.

Saida: query.


In [ ]:
%%spark

query = f'''
    SELECT
        mf1.CD_CLI,
        mf1.NR_AG_TITR,
        mf1.CD_CT_TITR,
        mf1.CD_PERFIL,
        mf1.PERFIL,
        mf1.CD_PRD,
        mf1.PRD,
        mf1.CD_CTGR_TRAN_OGNL,
        mf1.TX_DCR_CTGR_TRAN,
        (
            CASE
                WHEN LOCATE(mf1.LANCAMENTO, mf1.COMPLEMENTO) > 0
                    THEN mf1.LANCAMENTO
                ELSE REPLACE(
                    REPLACE(
                        REPLACE(
                            TRIM(mf1.LANCAMENTO) || ' ' || TRIM(mf1.COMPLEMENTO),
                            '  ',
                            ' '
                        ),
                        '  ',
                        ' '
                    ),
                    '  ',
                    ' '
                )
            END
        ) AS LANCAMENTO,
        mf1.TP_LANCAMENTO,
        AVG(mf1.VL_TRAN) AS MEDIA,
        SUM(mf1.VL_TRAN) AS SOMA,
        COUNT(mf1.VL_TRAN) AS QTD,
        mf1.DT_REF
    FROM (
        SELECT
            b.CD_CLI,
            b.NR_AG_TITR,
            b.CD_CT_TITR,
            b.CD_PERFIL,
            b.PERFIL,
            b.CD_PRD,
            b.PRD,

            TRIM(
                TRANSLATE(
                    (b.LANCAMENTO),
                    'oiauocaeaaaaaeeeeiiioooouuuc V()$&*%@;:_#',
                    'ÓÍÁÚÔÇÁÉãããããééééíõõõõúúúç1234567890/*-+.,| V()$&*%@;:_#'
                )
            ) AS LANCAMENTO,

            TRIM(
                TRANSLATE(
                    (b.COMPLEMENTO),
                    'oiauocaeaaaaaeeeeiiioooouuuc V()$&*%@;:_#',
                    'ÓÍÁÚÔÇÁÉãããããééééíõõõõúúúç1234567890/*-+.,| V()$&*%@;:_#'
                )
            ) AS COMPLEMENTO,

            b.CD_CTGR_TRAN_OGNL,

            TRIM(
                TRANSLATE(
                    c.TX_DCR_CTGR_TRAN,
                    'aaaeeiooouc',
                    'áãâéêíóôõúç'
                )
            ) AS TX_DCR_CTGR_TRAN,

            b.TP_LANCAMENTO,
            b.VL_TRAN,
            b.DT_REF

        FROM (
            SELECT
                a.CD_CLI,
                a.NR_AG_TITR,
                a.CD_CT_TITR,
                a.CD_PRD,
                (
                    CASE
                        WHEN a.CD_PRD = 6 THEN 'CONTA CORRENTE'
                        WHEN a.CD_PRD = 9 THEN 'CARTAO'
                    END
                ) AS PRD,
                a.CD_CTGR_TRAN_OGNL,
                LOWER(TRIM(a.TX_DCR_TRAN_OGNL)) AS LANCAMENTO,
                LOWER(TRIM(a.TX_CMPR_DCR_TRAN)) AS COMPLEMENTO,
                (
                    CASE
                        WHEN a.CD_NTZ_CTB_TRAN = 'C' THEN 'CREDITO'
                        WHEN a.CD_NTZ_CTB_TRAN = 'D' THEN 'DEBITO'
                    END
                ) AS TP_LANCAMENTO,
                a.VL_TRAN,
                cs.CD_SGM_CLI AS CD_PERFIL,
                (
                    CASE
                        WHEN cs.CD_SGM_CLI = 2502 THEN 'SEM PERFIL'
                        WHEN cs.CD_SGM_CLI = 2012 THEN 'PF A'
                        WHEN cs.CD_SGM_CLI = 2112 THEN 'PF B'
                        WHEN cs.CD_SGM_CLI = 2602 THEN 'PF C'
                        WHEN cs.CD_SGM_CLI = 2702 THEN 'PF D'
                        WHEN cs.CD_SGM_CLI = 2712 THEN 'PF E'
                        WHEN cs.CD_SGM_CLI = 1111 THEN 'A CLASSIFICAR'
                    END
                ) AS PERFIL,
                MONTH(a.DT_TRAN) || '/' || YEAR(a.DT_TRAN) AS DT_REF

            FROM DB2GFP.TRAN_RLZD_INST_PCT a
            INNER JOIN DB2SMI.CLI_SGM cs
                ON (a.CD_CLI = cs.CD_CLI)

            WHERE cs.CD_SGM_CLI IN (2502, 2012, 2112, 2602, 2702, 2712, 1111)
                AND cs.CD_CRIT_SGM_CLI = 2

                -- Seleção da partição
                AND a.NR_PTC BETWEEN 80 AND 85

                -- período de inclusão dos dados na tabela
                AND DATE(a.TS_INCL_TRAN) >= '2025-01-01'

                -- período das transações
                AND a.DT_TRAN BETWEEN '{data_ini}' AND '{data_fim}'
                AND a.CD_FST_TRAN_INST = 0
        ) b

        INNER JOIN DB2GFP.CTGR_TRAN_OPB c
            ON (b.CD_CTGR_TRAN_OGNL = c.CD_CTGR_TRAN)
    ) mf1

    GROUP BY
        mf1.CD_CLI,
        mf1.NR_AG_TITR,
        mf1.CD_CT_TITR,
        mf1.CD_PERFIL,
        mf1.PERFIL,
        mf1.CD_PRD,
        mf1.PRD,
        mf1.CD_CTGR_TRAN_OGNL,
        mf1.TX_DCR_CTGR_TRAN,
        mf1.LANCAMENTO,
        mf1.COMPLEMENTO,
        mf1.TP_LANCAMENTO,
        mf1.DT_REF
'''


# 07. Leitura JDBC transacoes

Objetivo: ler as transacoes no DB2 pelo cliente JDBC.

Entrada: query.

Saida: df_perfil.


In [ ]:
%%spark

df_perfil = cliente_db2.run_select(
    query,
    fetchsize=fetchsize_db2,
    partition_column="CD_CLI",
    lower_bound=cd_cli_min,
    upper_bound=cd_cli_max,
    num_partitions=num_partitions_transacoes
)


# 08. Leitura JDBC renda

Objetivo: ler somente as colunas necessarias de renda no DB2.

Entrada: DB2MCI.OCP_CFMD_BCRO.

Saida: df_ocp_cfmd_bcro.


In [ ]:
%%spark

df_ocp_cfmd_bcro = cliente_db2.run_select(
    """
    SELECT
        CD_CLI_OCP,
        TX_CRG_OCP,
        VL_REND_OCP,
        DT_REF_OCP,
        TS_INC_VGC_OCP
    FROM DB2MCI.OCP_CFMD_BCRO
    WHERE IN_OCP_PPL = 'S'
      AND DT_REF_OCP > DATE('2024-11-01')
    """,
    fetchsize=fetchsize_db2,
    partition_column="CD_CLI_OCP",
    lower_bound=cd_cli_min,
    upper_bound=cd_cli_max,
    num_partitions=num_partitions_db2
)


# 09. Transformacao renda

Objetivo: aplicar janela e agregacao de renda no Spark.

Entrada: df_ocp_cfmd_bcro.

Saida: df_renda_mf.


In [ ]:
%%spark

w_renda = Window.partitionBy("CD_CLI_OCP").orderBy(
    F.col("ts_inc_vgc_ocp").desc(),
    F.col("DT_REF_OCP").desc()
)

df_renda_mf = (
    df_ocp_cfmd_bcro
    .withColumn("ROW", F.row_number().over(w_renda))
    .where("ROW = 1")
    .select("CD_CLI_OCP", "TX_CRG_OCP", "VL_REND_OCP", "DT_REF_OCP")
    .groupBy("CD_CLI_OCP", "TX_CRG_OCP")
    .agg(F.max("VL_REND_OCP").alias("VL_REND"))
)


# 10. Join transacoes e renda

Objetivo: juntar transacoes com renda por cliente.

Entrada: df_perfil e df_renda_mf.

Saida: df_mfin.


In [ ]:
%%spark

df_mfin = df_renda_mf.join(
    df_perfil,
    F.expr("CD_CLI_OCP = CD_CLI"),
    "inner"
)


# 11. Leitura JDBC cliente

Objetivo: ler dados basicos de cliente no DB2.

Entrada: DB2MCI.CLIENTE.

Saida: df_mci.


In [ ]:
%%spark

df_mci = cliente_db2.run_select(
    """
    SELECT
        COD,
        DTA_NASC_CSNT
    FROM DB2MCI.CLIENTE
    """,
    fetchsize=fetchsize_db2,
    partition_column="COD",
    lower_bound=cd_cli_min,
    upper_bound=cd_cli_max,
    num_partitions=num_partitions_db2
)


# 12. Leitura JDBC pessoa fisica

Objetivo: ler dados cadastrais de pessoa fisica no DB2.

Entrada: DB2MCI.PESSOA_FISICA.

Saida: df_pf.


In [ ]:
%%spark

df_pf = cliente_db2.run_select(
    """
    SELECT
        F_CLIENTE_COD,
        COD_SEXO,
        COD_GRAU_INST,
        COD_ETDO_CVIL
    FROM DB2MCI.PESSOA_FISICA
    """,
    fetchsize=fetchsize_db2,
    partition_column="F_CLIENTE_COD",
    lower_bound=cd_cli_min,
    upper_bound=cd_cli_max,
    num_partitions=num_partitions_db2
)


# 13. Regras cadastrais

Objetivo: declarar expressoes de escolaridade e estado civil.

Entrada: df_pf.

Saida: cond_grau e cond_civil.


In [ ]:
%%spark

cond_grau = F.when(F.col("COD_GRAU_INST") == 1, F.lit('ANALFABETO')) \
    .when(F.col("COD_GRAU_INST") == 2, F.lit('ENSINO FUNDAMENTAL')) \
    .when(F.col("COD_GRAU_INST") == 3, F.lit('ENSINO MEDIO')) \
    .when(F.col("COD_GRAU_INST") == 4, F.lit('SUPERIOR INCOMPLETO')) \
    .when(F.col("COD_GRAU_INST") == 5, F.lit('SUPERIOR COMPLETO')) \
    .when(F.col("COD_GRAU_INST") == 6, F.lit('POS GRADUADO')) \
    .when(F.col("COD_GRAU_INST") == 7, F.lit('MESTRADO')) \
    .when(F.col("COD_GRAU_INST") == 8, F.lit('DOUTORADO')) \
    .when(F.col("COD_GRAU_INST") == 9, F.lit('SUPERIOR EM ANDAMENTO')) \
    .otherwise(F.lit("NÃO INFORMADO"))


cond_civil = F.when(F.col("COD_ETDO_CVIL") == 1, F.lit('Solteiro(a)')) \
    .when(F.col("COD_ETDO_CVIL") == 2, F.lit('Casado(a) - Comunhao Universal')) \
    .when(F.col("COD_ETDO_CVIL") == 3, F.lit('Casado(a) - Comunhao Parcial')) \
    .when(F.col("COD_ETDO_CVIL") == 4, F.lit('Casado(a) - Separacao de Bens')) \
    .when(F.col("COD_ETDO_CVIL") == 5, F.lit('Viuvo(a)')) \
    .when(F.col("COD_ETDO_CVIL") == 6, F.lit('Separado(a) Judicialmente')) \
    .when(F.col("COD_ETDO_CVIL") == 7, F.lit('Divorciado(a)')) \
    .when(F.col("COD_ETDO_CVIL") == 8, F.lit('Casado(a) - Regime Total')) \
    .when(F.col("COD_ETDO_CVIL") == 9, F.lit('Casado(a) - Regime Misto/Especial')) \
    .when(F.col("COD_ETDO_CVIL") == 10, F.lit('Uniao Estavel')) \
    .otherwise(F.lit("Não Informado"))


# 14. Montagem publico

Objetivo: montar o publico enriquecido com renda e cadastro.

Entrada: df_mfin, df_mci, df_pf, cond_grau e cond_civil.

Saida: df_publico.


In [ ]:
%%spark

df_publico = df_mfin.alias("mfin") \
    .join(
        df_mci.alias("cli"),
        F.expr("mfin.cd_cli = cli.cod"),
        'inner'
    ) \
    .join(
        df_pf.alias("pf"),
        F.expr("mfin.cd_cli = pf.F_CLIENTE_COD"),
        'inner'
    ) \
    .withColumn("Idade", ano_etl - F.year("dta_nasc_csnt")) \
    .withColumn("GRAU_INST", cond_grau) \
    .withColumn("ETDO_CIVIL", cond_civil) \
    .select(
        "CD_CLI",
        "NR_AG_TITR",
        "CD_CT_TITR",
        "CD_PERFIL",
        "PERFIL",
        "IDADE",
        "COD_SEXO",
        "GRAU_INST",
        "VL_REND",
        "TX_CRG_OCP",
        "ETDO_CIVIL",
        "CD_PRD",
        "PRD",
        "CD_CTGR_TRAN_OGNL",
        "TX_DCR_CTGR_TRAN",
        "LANCAMENTO",
        "TP_LANCAMENTO",
        "SOMA",
        "MEDIA",
        "QTD",
        "DT_REF"
    )


# 15. Schema Hive destino

Objetivo: ler o contrato de colunas da tabela destino Hive.

Entrada: nome_tabela.

Saida: prod_cols.


In [ ]:
%%spark

prod_cols = spark.table(nome_tabela).columns


# 16. Montagem tabela final

Objetivo: aplicar renomes, chave, filtros finais e TS_ATL.

Entrada: df_publico e prod_cols.

Saida: df_final.


In [ ]:
%%spark

w_final = Window.partitionBy("dt_ref").orderBy("SOMA", "CD_CLI")

df_final = (
    df_publico
    .where("CD_CLI IS NOT NULL")
    .withColumn("CD_CT_TITR", F.col("CD_CT_TITR").cast("integer"))
    .withColumn("DT_REF", F.to_date("dt_ref", "M/yyyy"))
    .withColumn("ROW", F.row_number().over(w_final))
    .where("DT_REF IS NOT NULL")
    .withColumn("CD_CHV_REG", F.sha1(F.concat("CD_CLI", "dt_ref", "soma", "ROW")))
    .drop("row")
    .withColumn("AA_MM_REF", F.date_format("DT_REF", "yyyyMM"))
    .withColumnRenamed("CD_PERFIL", "CD_PRFL")
    .withColumnRenamed("PERFIL", "TX_PRFL")
    .withColumnRenamed("IDADE", "NR_IDD_CLI")
    .withColumnRenamed("COD_SEXO", "CD_SEXO")
    .withColumnRenamed("GRAU_INST", "TX_DCR_NVL_ITC")
    .withColumnRenamed("PERFIL", "TX_PRFL")
    .withColumnRenamed("VL_REND", "VL_REN")
    .withColumnRenamed("ETDO_CIVIL", "TX_EST_CVL")
    .withColumnRenamed("PRD", "TX_DCR_PRD")
    .withColumnRenamed("TX_DCR_CTGR_TRAN", "TX_DCR_CTGR_TRNS")
    .withColumnRenamed("lancamento", "tx_anot_ctgr_trns")
    .withColumnRenamed("tp_lancamento", "tx_tip_trns")
    .withColumnRenamed("media", "vl_med_trns")
    .withColumnRenamed("soma", "vl_ttl_trns")
    .withColumnRenamed("QTD", "QT_TRNS")
    .withColumnRenamed("CD_CTGR_TRAN_OGNL", "cd_ctgr_trns_ognl")
    .withColumn("LEN", F.length(F.col("tx_anot_ctgr_trns")))
    .where("Len < 200")
)

ts_atl = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

if "TS_ATL" not in df_final.columns:
    df_final = df_final.withColumn(
        "TS_ATL",
        F.lit(ts_atl).cast("timestamp")
    )

df_final = df_final.select(*prod_cols)


# 17. Diagnostico opt-in

Objetivo: permitir inspecao opcional de schemas.

Entrada: dataframes principais.

Saida: schemas impressos somente quando habilitado.


In [ ]:
%%spark

if diagnostico_opt_in:
    df_perfil.printSchema()
    df_ocp_cfmd_bcro.printSchema()
    df_renda_mf.printSchema()
    df_mfin.printSchema()
    df_mci.printSchema()
    df_pf.printSchema()
    df_publico.printSchema()
    df_final.printSchema()


# 18. Publicacao Hive

Objetivo: gravar o dataframe final na tabela Hive de destino.

Entrada: df_final e nome_tabela.

Saida: tabela Hive atualizada.


In [ ]:
%%spark

(
    df_final
    .coalesce(partitions_hive)
    .write
    .mode("overwrite")
    .insertInto(nome_tabela)
)

# 19. Cleanup

Objetivo: encerrar a sessao usando o mecanismo ja existente.

Entrada: sessao Spark ativa.

Saida: sessao encerrada.


In [ ]:
%spark cleanup
